# 00 — Probes before the data pipeline

One-off checks the data-generation design depends on (`hvac_controller/probes.py`):
step time and CPU count, determinism, mid-episode snapshot, the AHU-off defect, scenario-generator
reproducibility, and a scenario built from a TMYx station of the weather dataset.

**Setup:** Internet **on** (to clone the repo); attach the competition data and the weather dataset
(`data.zip`). CPU accelerator. Expected run time ≈ 10–20 min.

**Parallel-run check:** Save Version with `RUN_ID = 0`, then immediately set `RUN_ID = 1` and Save
Version again. If both versions finish (not *cancelled*) and their `start`/`end` times overlap,
one notebook can run several `RUN_ID`s at once.

Output: `/kaggle/working/probe_run<RUN_ID>.zip` with `probe_report.json`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "curr"  # working branch; main is never touched
RUN_ID = 0         # change before each Save Version

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

START = time.strftime("%Y-%m-%d %H:%M:%S")
REPO = Path("/tmp/hvac-controller")  # outside /kaggle/working, so the clone is not saved as output
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
os.chdir(REPO)  # scenario configs hold repo-relative paths
sys.path.insert(0, str(REPO))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
INPUT = Path("/kaggle/input")
# The shipped scenarios are git-ignored in the repo; take them from the competition data.
SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)

from hvac_controller import probes, weather

try:
    PACKED = weather.find_packed(INPUT)
except FileNotFoundError:
    PACKED = None
print("shipped scenarios:", SHIPPED)
print("packed weather   :", PACKED)

In [ ]:
report = probes.run_all(REPO, SHIPPED, PACKED, Path("/tmp/probe_work"))
report["run"] = {"run_id": RUN_ID, "repo_ref": REPO_REF, "start": START, "end": time.strftime("%Y-%m-%d %H:%M:%S")}

out = Path("/kaggle/working") / f"probe_run{RUN_ID}"
out.mkdir(parents=True, exist_ok=True)
(out / "probe_report.json").write_text(json.dumps(report, indent=1, default=str))
shutil.make_archive(str(out), "zip", out)
shutil.rmtree(out)
print("wrote", f"{out}.zip")

In [ ]:
# Compact summary (the full report is in the zip).
def show(name, keys):
    r = report.get(name, {})
    if "error" in r:
        print(f"{name:12s} ERROR {r['error']}")
    else:
        print(f"{name:12s}", {k: r.get(k) for k in keys})

show("environment", ["cpu_count", "sched_affinity", "python", "numpy", "repo_commit"])
show("step_time", ["seconds_per_step", "projected_minutes_8640_steps", "n_ahu", "n_zone", "obs_dim"])
show("determinism", ["identical_after_reset", "identical_fresh_env", "same_env_after_reset", "fresh_env"])
show("snapshot", ["deepcopy_seconds", "second_snapshot_seconds", "shares_factorization", "identical",
                  "identical_second_copy", "branch_seconds_per_step", "original_vs_copy"])
show("ahu_off", ["warm_steps", "branch_at", "summary"])
show("generator", ["seconds", "identical"])
show("new_weather", ["station", "convection_coefficient", "rewards_finite", "max_abs_temp_error_k", "steps"])
print("run", report["run"])